# META-PM POC Analysis

This notebook validates Polymarket data ingestion and engine PnL calculations.


## Cell 1: Inspect raw Polymarket prices

Load and inspect price data for validation:
- M3 (YES outcome market) - should converge toward 1.0
- M1 (NO outcome market) - should converge toward 0.0


In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Load M3 (YES outcome market)
prices_m3 = pd.read_csv("data/prices_M3.csv", parse_dates=["timestamp"])
print("M3 - US inflation >0.1% June→July 2024 (YES outcome)")
print(prices_m3.head())
print("\nPrice statistics:")
print(prices_m3["price"].describe())

# Load M1 (NO outcome market)
prices_m1 = pd.read_csv("data/prices_M1.csv", parse_dates=["timestamp"])
print("\n\nM1 - US inflation >0.3% March→April 2024 (NO outcome)")
print(prices_m1.head())
print("\nPrice statistics:")
print(prices_m1["price"].describe())


## Cell 2: Plot price time series to verify convergence

Plot price paths to verify they converge toward correct outcomes:
- M3 (YES) should move toward 1.0 near resolution
- M1 (NO) should move toward 0.0 near resolution


In [ ]:
# Plot M3 (YES outcome market)
plt.figure(figsize=(12, 5))
plt.plot(prices_m3["timestamp"], prices_m3["price"], label="M3 price")
plt.xlabel("Time")
plt.ylabel("Implied YES probability")
plt.title("M3 – US inflation >0.1% June→July 2024 (YES outcome)")
plt.grid(True)
plt.legend()
plt.show()

# Plot M1 (NO outcome market)
plt.figure(figsize=(12, 5))
plt.plot(prices_m1["timestamp"], prices_m1["price"], label="M1 price", color="orange")
plt.xlabel("Time")
plt.ylabel("Implied YES probability")
plt.title("M1 – US inflation >0.3% March→April 2024 (NO outcome)")
plt.grid(True)
plt.legend()
plt.show()


## Cell 3: Load engine outputs

Load trades and scores from engine backtest.


In [ ]:
trades = pd.read_csv("outputs/trades.csv", parse_dates=["entry_time"])
scores = pd.read_csv("outputs/scores.csv")

print("Trades:")
display(trades.head())
print(f"\nTotal trades: {len(trades)}")
print("\nScores:")
display(scores)


## Cell 4: Plot cumulative PnL per strategy

Plot equity curves to see strategy performance over time.


In [ ]:
equity_curves = {}

for strat_name, df in trades.groupby("strategy_name"):
    df_sorted = df.sort_values("entry_time")
    df_sorted["cum_pnl"] = df_sorted["pnl"].cumsum()
    equity_curves[strat_name] = df_sorted

plt.figure(figsize=(12, 6))
for strat_name, df in equity_curves.items():
    plt.plot(df["entry_time"], df["cum_pnl"], label=strat_name, marker="o", markersize=3)

plt.xlabel("Time")
plt.ylabel("Cumulative PnL")
plt.title("Cumulative PnL per strategy on Polymarket markets")
plt.legend()
plt.grid(True)
plt.show()


## Cell 5: Show leaderboard sorted by total_pnl

Quick leaderboard view of strategy performance.


In [6]:
# Load scores from both built-in strategies and plugin strategies
scores_builtin = pd.read_csv("outputs/scores.csv")
scores_plugins = pd.read_csv("outputs/scores_from_logs.csv")

# Combine both dataframes
scores = pd.concat([scores_builtin, scores_plugins], ignore_index=True)

scores_sorted = scores.sort_values("total_pnl", ascending=False)
print("Strategy Leaderboard (sorted by Total P&L):")
print("=" * 70)
print(f"Built-in strategies: {len(scores_builtin)}, Plugin strategies: {len(scores_plugins)}")
print("=" * 70)
display(scores_sorted)

Strategy Leaderboard (sorted by Total P&L):
Built-in strategies: 4, Plugin strategies: 5


,strategy_name,total_pnl,brier_mean,max_drawdown,n_trades
6,mean_revert_v1,31.075,0.180990,15.15,38
3,oracle,14.700,0.000100,0.00,5
2,filtered_momentum,0.000,NaN,NaN,0
1,momentum,-1.250,0.195881,7.00,4
7,cross_section_value_v1,-1.730,0.017353,1.73,3
8,risk_managed_momo_v1,-4.400,0.207318,8.25,11
0,price_follower,-7.300,0.138420,11.00,5
4,baseline_hold,-7.300,0.138420,11.00,5
5,swing_v1,-10.400,0.185595,21.40,27


## Cell 6: Calibration check (optional)

Scatter plot of p_hat vs outcome to check calibration.
Higher p_hat should cluster with outcome=1, lower p_hat with outcome=0.


## Cell 7: Per-market P&L analysis

Analyze strategy performance broken down by market to see where profits/losses come from.


In [7]:
# Load trades if not already loaded
if 'trades' not in globals():
    trades = pd.read_csv("outputs/trades.csv", parse_dates=["entry_time"])

# Per-strategy, per-market total PnL and trade counts
pnl_by_market = (
    trades
    .groupby(["strategy_name", "market_id"])
    .agg(
        total_pnl=("pnl", "sum"),
        n_trades=("pnl", "count"),
        avg_entry_price=("entry_price", "mean"),
    )
    .reset_index()
    .sort_values(["strategy_name", "market_id"])
)

print("Per-market P&L breakdown:")
display(pnl_by_market)


Per-market P&L breakdown:


,strategy_name,market_id,total_pnl,n_trades,avg_entry_price
0,momentum,M1,-7.00,1,0.700
1,momentum,M2,2.20,1,0.780
2,momentum,M3,4.60,1,0.540
3,momentum,M4,-1.05,1,0.105
4,oracle,M1,5.25,1,0.525
5,oracle,M2,2.85,1,0.715
6,oracle,M3,5.75,1,0.425
7,oracle,M4,0.65,1,0.065
8,oracle,M5,0.20,1,0.980
9,price_follower,M1,-5.25,1,0.525


In [8]:
# Pivot view: strategy vs market P&L matrix
pnl_pivot = pnl_by_market.pivot(
    index="strategy_name", columns="market_id", values="total_pnl"
).fillna(0.0)

print("Strategy vs Market P&L Matrix:")
print("=" * 70)
display(pnl_pivot)

# Show which markets each strategy is best/worst on
print("\nBest and worst market for each strategy:")
for strat in pnl_pivot.index:
    row = pnl_pivot.loc[strat]
    best_market = row.idxmax()
    worst_market = row.idxmin()
    print(f"{strat:20s} - Best: {best_market} ({row[best_market]:+.2f}), Worst: {worst_market} ({row[worst_market]:+.2f})")


Strategy vs Market P&L Matrix:


market_id,M1,M2,M3,M4,M5
strategy_name,,,,,
momentum,-7.00,2.20,4.60,-1.05,0.0
oracle,5.25,2.85,5.75,0.65,0.2
price_follower,-5.25,2.85,-5.75,0.65,0.2



Best and worst market for each strategy:
momentum             - Best: M3 (+4.60), Worst: M1 (-7.00)
oracle               - Best: M3 (+5.75), Worst: M5 (+0.20)
price_follower       - Best: M2 (+2.85), Worst: M3 (-5.75)


## Cell 8: Load per-market scores from engine output

Load the per-market scores CSV files generated by the engine.


In [9]:
# Load per-market scores from both built-in and plugin strategies
scores_by_m_builtin = pd.read_csv("outputs/scores_by_market.csv")
scores_by_m_plugins = pd.read_csv("outputs/scores_by_market_from_logs.csv")

# Combine both dataframes
scores_by_m = pd.concat([scores_by_m_builtin, scores_by_m_plugins], ignore_index=True)

print("Per-market strategy scores (combined):")
print("=" * 70)
display(scores_by_m.sort_values(["strategy_name", "market_id"]))

# Create pivot table for strategy vs market P&L
pnl_pivot = scores_by_m.pivot(
    index="strategy_name", columns="market_id", values="total_pnl"
).fillna(0.0)

print("\nStrategy vs Market P&L Matrix (from engine output):")
print("=" * 70)
display(pnl_pivot)


Per-market strategy scores (combined):


,strategy_name,market_id,total_pnl,brier_mean,max_drawdown,n_trades
14,baseline_hold,M1,-5.250,0.275625,5.250,1
15,baseline_hold,M2,2.850,0.081225,0.000,1
16,baseline_hold,M3,-5.750,0.330625,5.750,1
17,baseline_hold,M4,0.650,0.004225,0.000,1
18,baseline_hold,M5,0.200,0.000400,0.000,1
27,cross_section_value_v1,M2,-1.650,0.046225,1.650,1
28,cross_section_value_v1,M4,-0.050,0.003025,0.050,1
29,cross_section_value_v1,M5,-0.030,0.002809,0.030,1
23,mean_revert_v1,M1,30.300,0.435068,15.150,11
24,mean_revert_v1,M2,2.050,0.093611,10.800,11



Strategy vs Market P&L Matrix (from engine output):


market_id,M1,M2,M3,M4,M5
strategy_name,,,,,
baseline_hold,-5.250,2.850,-5.75,0.650,0.20
cross_section_value_v1,0.000,-1.650,0.00,-0.050,-0.03
mean_revert_v1,30.300,2.050,-5.25,3.975,0.00
momentum,-7.000,2.200,4.60,-1.050,0.00
oracle,5.250,2.850,5.75,0.650,0.20
price_follower,-5.250,2.850,-5.75,0.650,0.20
risk_managed_momo_v1,1.525,-4.625,1.50,-2.800,0.00
swing_v1,-5.750,-4.300,3.00,-3.350,0.00


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(trades["p_hat"], trades["outcome"], alpha=0.4, s=50)
plt.xlabel("p_hat (strategy probability)")
plt.ylabel("Outcome (0/1)")
plt.title("p_hat vs outcome across all Polymarket trades")
plt.grid(True)
plt.show()

# Summary statistics by outcome
print("Mean p_hat by outcome:")
print(trades.groupby("outcome")["p_hat"].mean())
